In [1]:
import os
from pathlib import Path

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
env_path = Path("../.env")
if env_path.exists():
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("\"'")
        os.environ.setdefault(key, value)

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in ../.env or environment before running notebook.")

In [2]:
from peft import get_peft_model, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)

In [5]:
model.print_trainable_parameters()

trainable params: 4,587,520 || all params: 756,219,904 || trainable%: 0.6066


In [ ]:
from trl import SFTConfig, SFTTrainer

train_cfg = SFTConfig(
    output_dir="../outputs",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=20,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    max_seq_length=1024,
    bf16=True,
    report_to=['tensorboard'],
)

In [6]:
from datasets import load_dataset
ds = load_dataset("../data/alpaca_gpt4")

In [9]:
ds['validation']['text']

Column(["Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat would be the best type of exercise for a person who has arthritis?\n\n### Response:\nIf a person has arthritis, low-impact exercises that are gentle on their joints are the best options. Some effective forms of low-impact exercises include:\n\n1. Walking: Walking is a simple, low-impact exercise that can help improve strength, balance and joint flexibility. \n\n2. Water exercises: Swimming or performing water aerobics are great ways to exercise in a weightless environment, which reduces pressure and strain on the joints. \n\n3. Yoga: Practicing yoga is a good way to gently stretch the muscles, improve flexibility and balance, and reduce joint stiffness. \n\n4. Cycling: Stationary cycling is another low-impact way to exercise, as it takes the weight off the joints, and can improve cardiovascular health, as well as leg strength. \n\n5. Tai Chi: Tai 

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=train_cfg,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    dataset_text_field="text",
)
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
